In [1]:
import sys
import os
import json
sys.path.append(os.path.abspath('../'))
from src.data_loader import DataLoader
from src.preprocessing import clean_text, stem_text, split_data
from src.train import train_model
from src.evaluate import test_model
import matplotlib.pyplot as plt
from nltk.stem.snowball import SnowballStemmer
import src.config as config

loader = DataLoader(config.RAW_DATA_PATH)
df = loader.get_clean_ticket_dataset()

with open(config.LANGUAGES_PATH, 'r', encoding='utf-8') as file:
    LANGUAGES = json.load(file)

df['cleaned_context'] = df['context_problem'].apply(clean_text)

df['stemmed_context'] = df['cleaned_context']
for lang_code, lang_name in LANGUAGES.items():
    stemmer = SnowballStemmer(lang_name)
    df.loc[df['language'] == lang_code, 'stemmed_context'] = df.loc[df['language'] == lang_code, 'cleaned_context'].apply(lambda x: stem_text(x, stemmer))

TRAINING_COL = 'stemmed_context'
TARGET_COL = 'queue'

train_df, test_df = split_data(df=df, target_column=TARGET_COL)

train_df.to_csv('..\\data\\processed\\train.csv', index=False)
test_df.to_csv('..\\data\\processed\\test.csv', index=False)

print(f"Train and test datasets saved. Train: {train_df.shape[0]} rows, Test: {test_df.shape[0]} rows.")

print(" --- Starting model training --- ")

train_model(training_col=TRAINING_COL, target_col=TARGET_COL, languages=list(LANGUAGES.values()))

print(" --- Starting model evaluation --- ")
print(" --- Evaluating on test set --- ")

test_model(training_col=TRAINING_COL, target_col=TARGET_COL)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\szymon2256\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
2026-05-22 17:26:12,058 - INFO -  Data loaded successfully from c:\Projects\ai-ml-engineering-portfolio\01-nlp-baseline-ticket-classifier\data\raw\tickets_dataset.csv
2026-05-22 17:26:12,671 - INFO -  Data cleaned successfully
2026-05-22 17:26:16,351 - INFO - Splitting data into train and test sets with test size 0.2 and random state 42...
2026-05-22 17:26:16,363 - INFO - Data splitting completed.
2026-05-22 17:26:16,544 - INFO - Loading training data...
2026-05-22 17:26:16,713 - INFO - Initializing the model pipeline with TF-IDF and Logistic Regression...
2026-05-22 17:26:16,713 - INFO - Training the model on 3200 rows...


Train and test datasets saved. Train: 3200 rows, Test: 800 rows.
 --- Starting model training --- 


2026-05-22 17:26:17,921 - INFO - Model training completed.
2026-05-22 17:26:17,923 - INFO - Saving the trained model to c:\Projects\ai-ml-engineering-portfolio\01-nlp-baseline-ticket-classifier\models\baseline_pipeline.joblib...
2026-05-22 17:26:18,169 - INFO - Model saved successfully.
2026-05-22 17:26:18,175 - INFO - Loading the test data...
2026-05-22 17:26:18,211 - INFO - Loading the trained model from c:\Projects\ai-ml-engineering-portfolio\01-nlp-baseline-ticket-classifier\models\baseline_pipeline.joblib...
2026-05-22 17:26:18,320 - INFO - Model loaded successfully. Making predictions on the test set...


 --- Starting model evaluation --- 
 --- Evaluating on test set --- 


2026-05-22 17:26:18,832 - INFO - Confusion matrix plot saved to c:\Projects\ai-ml-engineering-portfolio\01-nlp-baseline-ticket-classifier\reports\figures\confusion_matrix.png
2026-05-22 17:26:18,842 - INFO - Evaluation report generated and saved to c:\Projects\ai-ml-engineering-portfolio\01-nlp-baseline-ticket-classifier\reports\baseline_report.md


({'Accuracy': 0.57375,
  'Macro F1': 0.5221740278573404,
  'Weighted F1': 0.5651363535184066},
 array([[ 64,   2,   1,   0,   0,   0,   0,   0,   0,   1],
        [  1,  59,   1,   0,  14,  19,   3,   2,   1,  25],
        [  0,   6,   1,   0,   2,   0,   0,   0,   2,   0],
        [  0,   1,   0,   4,   1,   1,   0,   0,   0,   4],
        [  2,   9,   0,   0,  25,   7,   3,   2,   1,  40],
        [  5,  21,   0,   1,   6,  70,   2,   3,   1,  29],
        [  1,   4,   0,   0,   1,   2,  30,   0,   0,   1],
        [  1,   6,   1,   0,   1,   3,   0,  11,   0,   4],
        [  1,   0,   0,   0,   0,   0,   0,   0,  15,  12],
        [  1,  21,   1,   0,  22,  27,   5,   1,   6, 180]]))